# 02 Preprocess - Sample Or Full Raw Files

Use `RUN_MODE = "sample"` for local smoke runs and `RUN_MODE = "full"` on the server. Sample mode writes a separate parquet artifact and does not overwrite the full 2M processed file.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import pandas as pd
sys.path.append('..')

from src.full_pipeline import (
    RAW_DIR,
    PROCESSED_PATH,
    preprocess_full_raw_to_parquet,
    preprocess_raw_sample_to_parquet,
    sample_processed_path,
    parquet_row_count,
)


## 1. Preprocess Configuration


In [ ]:
RUN_MODE = "sample"  # "sample" for local, "full" for server
ROWS_PER_CATEGORY = 1_000
RAW_CHUNKSIZE = 25_000 if RUN_MODE == "sample" else 100_000
MIN_LENGTH = 10
MAX_TOKENS = 256
DEDUPLICATE = True
OVERWRITE_PROCESSED = True

ACTIVE_PROCESSED_PATH = sample_processed_path(ROWS_PER_CATEGORY) if RUN_MODE == "sample" else PROCESSED_PATH

print(f"Run mode: {RUN_MODE}")
print(f"Output parquet: {ACTIVE_PROCESSED_PATH}")
if RUN_MODE == "sample":
    print(f"Rows per category target: {ROWS_PER_CATEGORY:,}")
else:
    print("This will process every row from every data/raw/*.jsonl file.")


## 2. Run Raw -> Processed Parquet


In [ ]:
if RUN_MODE == "sample":
    summary = preprocess_raw_sample_to_parquet(
        raw_dir=RAW_DIR,
        output_path=ACTIVE_PROCESSED_PATH,
        rows_per_category=ROWS_PER_CATEGORY,
        chunksize=RAW_CHUNKSIZE,
        min_length=MIN_LENGTH,
        max_tokens=MAX_TOKENS,
        deduplicate=DEDUPLICATE,
        overwrite=OVERWRITE_PROCESSED,
    )
else:
    summary = preprocess_full_raw_to_parquet(
        raw_dir=RAW_DIR,
        output_path=ACTIVE_PROCESSED_PATH,
        chunksize=RAW_CHUNKSIZE,
        min_length=MIN_LENGTH,
        max_tokens=MAX_TOKENS,
        deduplicate=DEDUPLICATE,
        overwrite=OVERWRITE_PROCESSED,
    )

for key in ["mode", "raw_rows", "raw_rows_scanned", "kept_rows", "dropped_empty_or_short", "dropped_duplicates"]:
    if key in summary:
        print(f"{key}: {summary[key]:,}" if isinstance(summary[key], int) else f"{key}: {summary[key]}")

print("Kept rows by category:")
for category, count in summary['kept_rows_by_category'].items():
    print(f"- {category}: {count:,}")


## 3. Validate Processed Artifact


In [ ]:
row_count = parquet_row_count(ACTIVE_PROCESSED_PATH)
print(f"Processed parquet rows: {row_count:,}")

preview = pd.read_parquet(ACTIVE_PROCESSED_PATH, columns=['rating', 'category', 'title', 'cleaned_text']).head(5)
display(preview)
